# 06 — Benchmark: base vs. fine-tuned YourMT3+ variants

Standalone benchmarking notebook — **run on a fresh Colab GPU runtime, no training
required**. It picks a finished training run (`comparison_<ts>` on Drive, produced by
notebook 05), restores the base + fine-tuned checkpoints from Drive, benchmarks every
model on every category with **note-level F1** (mir_eval, onset ±50 ms), and renders
the comparison charts. All results land in the run's Drive folder as they are produced.

Categories: `strudel_corpus` (true test) · 4 synthetic batches (**val-diag** — scored on
their validation files; read as synth-timbre diagnostic, not honest generalization) ·
maestro / slakh / egmd (reference sets, capped at `REF_CAP`).


## 1. Imports, Google Drive, environment


In [ ]:
# Everything import-y lives here.
import json, os, re, sys, csv, time, shutil, subprocess, datetime
from pathlib import Path
from collections import defaultdict, OrderedDict

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/restrudel")
    if not Path("/content/restrudel").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/henrik253/Restrudel.git", "/content/restrudel"], check=True)
    REPO = Path("/content/restrudel")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
else:
    REPO = Path(os.environ.get("RESTRUDEL_REPO", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
    DRIVE = Path(os.environ.get("RESTRUDEL_DRIVE", REPO))

DATA_HOME = DRIVE / "datasets"
INDEX_DIR = DATA_HOME / "yourmt3_indexes"
MODEL_ROOT = REPO / "models" / "YourMT3"            # cwd for train.py / test.py
AMT_SRC = MODEL_ROOT / "amt" / "src"
DRIVE_MODEL_CACHE = DRIVE / "models" / "YourMT3"    # persistent base-model cache

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
try:
    import torch
    GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except Exception:
    torch, GPU = None, None

print(f"colab={IN_COLAB}  repo={REPO}")
print(f"data_home={DATA_HOME}")
print(f"GPU={GPU}")
assert INDEX_DIR.exists(), "no index dir — build the datasets in Drive (notebook 04) first"


## 2. Configuration

Pick **which training run to benchmark** (`COMPARISON_TS = ""` → the latest
`comparison_*` folder on Drive). Fine-tuned variants are discovered from their
`YourMT3+_fine_tuned_<variant>_<ts>` Drive folders (`metadata.json` + `last.ckpt`);
unfinished variants are skipped automatically.


In [ ]:
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

# --- which training run to benchmark ------------------------------------------
COMPARISON_TS = ""                       # "" -> latest comparison_* dir on Drive
CKPT_ROOT = DRIVE / "checkpoints"
if COMPARISON_TS:
    COMPARE_DIR = CKPT_ROOT / f"comparison_{COMPARISON_TS}"
    assert COMPARE_DIR.exists(), f"{COMPARE_DIR} not found"
else:
    _dirs = sorted(d for d in CKPT_ROOT.glob("comparison_*") if d.is_dir())
    assert _dirs, "no comparison_* run dirs on Drive — train with notebook 05 first"
    COMPARE_DIR = _dirs[-1]
RUN_TS = COMPARE_DIR.name.replace("comparison_", "")
RUN_NAME = COMPARE_DIR.name
print("benchmarking training run:", RUN_NAME)

# --- fine-tuned variants of that run (from Drive) ------------------------------
VARIANTS = OrderedDict()
for d in sorted(CKPT_ROOT.glob(f"YourMT3+_fine_tuned_*_{RUN_TS}")):
    meta = json.load(open(d / "metadata.json")) if (d / "metadata.json").exists() else {}
    if (d / "last.ckpt").exists() and meta.get("status") == "done":
        VARIANTS[meta["variant"]] = {"exp": meta["exp"], "dir": d, "weights": meta.get("weights")}
    else:
        print(f"  skipping {d.name}: status={meta.get('status')!r} ckpt={(d / 'last.ckpt').exists()}")
assert VARIANTS, "no finished fine-tuned variants found for this run"
print("variants:", {v: c["exp"] for v, c in VARIANTS.items()})

# --- categories (same taxonomy as notebook 05) ---------------------------------
SEED2BATCH = {"1": "batch_1", "1000": "batch_2", "2000": "batch_3", "3000": "batch_4", "4000": "batch_5"}
CATEGORIES = OrderedDict([
    ("strudel_corpus",            ("Strudel · corpus",   "target",    "electronic", "#d1495b")),
    ("strudel_synthetic_batch_1", ("Strudel · synth b1", "target",    "electronic", "#e36414")),
    ("strudel_synthetic_batch_2", ("Strudel · synth b2", "target",    "electronic", "#e8871e")),
    ("strudel_synthetic_batch_3", ("Strudel · synth b3", "target",    "electronic", "#f3a712")),
    ("strudel_synthetic_batch_4", ("Strudel · synth b4", "target",    "electronic", "#f7c548")),
    ("maestro",                   ("MAESTRO (piano)",    "reference", "acoustic",   "#66a182")),
    ("slakh",                     ("Slakh (band)",       "reference", "acoustic",   "#2e4057")),
    ("egmd",                      ("EGMD (e-drums)",     "reference", "electronic", "#8d99ae")),
])
STRUDEL_SUBCATS = [c for c in CATEGORIES if c.startswith("strudel")]

def strudel_subcat(sid):
    if sid.startswith("corpus_"):
        return "strudel_corpus"
    m = re.match(r"inspired_(\d+)_\d+$", sid)          # inspired_<seed>_<i> (batch 2+)
    if m:
        return "strudel_synthetic_" + SEED2BATCH.get(m.group(1), "seed" + m.group(1))
    if re.match(r"inspired_\d+$", sid):                # inspired_<NNN> (seed 1 = batch 1)
        return "strudel_synthetic_batch_1"
    return "strudel_corpus"

# WEIGHTS is only referenced inside the preset block shared with notebook 05
# (the strudel_ft multi preset); test.py never samples with it.
WEIGHTS = OrderedDict([("strudel", 0.50), ("slakh", 0.25), ("maestro", 0.15), ("egmd", 0.10)])

# --- eval scope ----------------------------------------------------------------
FULL_EVAL = False          # True = whole test splits (slow!)
EVAL_CAP  = 200            # cap per strudel category
REF_CAP   = 50             # cap per reference set (maestro/slakh/egmd) — they only
                           # sanity-check forgetting; 50 files ≈ ±2-3 pt F1 noise
PRECISION = "bf16-mixed" if (torch and torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "16-mixed"

# architecture flags — MUST match the released checkpoint
ARCH = ["-tk", "mc13_full_plus_256", "-dec", "multi-t5", "-nl", "26",
        "-enc", "perceiver-tf", "-sqr", "1", "-ff", "moe", "-wf", "4",
        "-nmoe", "8", "-kmoe", "2", "-act", "silu", "-epe", "rope", "-rp", "1",
        "-ac", "spec", "-hop", "300", "-atc", "1"]
RELEASED_EXP = "mc13_256_g4_all_v7_mt3f_sqr_rms_moe_wf4_n8k2_silu_rope_rp_b36_nops"

MODELS = OrderedDict([("base", (RELEASED_EXP, "2024"))]
                     + [(v, (VARIANTS[v]["exp"], "ymt3")) for v in VARIANTS])
print(f"models={list(MODELS)}  eval_cap={EVAL_CAP}  ref_cap={REF_CAP}  precision={PRECISION}")

## 3. Base model — code + inference deps (Drive-cached)

Same setup as notebook 05 §3: fetch YourMT3's `amt/src` (Drive cache), install the
dependency stack, and apply the compatibility patches (idempotent). Benchmarking only
*needs* the inference-side ones — transformers pin, `torch.load`, torch.compile off,
Drive I/O retry — but applying the full identical set keeps both notebooks in lockstep.


In [ ]:
from huggingface_hub import snapshot_download

def ensure_code():
    if (AMT_SRC / "config" / "data_presets.py").exists():
        print("model code already local:", AMT_SRC); return
    cache_src = DRIVE_MODEL_CACHE / "amt" / "src"
    if (cache_src / "config" / "data_presets.py").exists():
        print("restoring model code from Drive cache…")
        shutil.copytree(DRIVE_MODEL_CACHE / "amt" / "src", AMT_SRC, dirs_exist_ok=True)
    else:
        print("fetching model code from HF Space (one-time)…")
        snapshot_download(repo_id="mimbres/YourMT3", repo_type="space",
                          allow_patterns=["amt/src/**", "amt/src/*"], local_dir=str(MODEL_ROOT))
        (DRIVE_MODEL_CACHE / "amt").mkdir(parents=True, exist_ok=True)
        shutil.copytree(AMT_SRC, DRIVE_MODEL_CACHE / "amt" / "src", dirs_exist_ok=True)
    assert (AMT_SRC / "config" / "data_presets.py").exists(), "model code missing after ensure_code()"
    print("model code ready:", AMT_SRC)

ensure_code()

# --- YourMT3 training deps. transformers PINNED <4.40 (newer removed
#     transformers.utils.model_parallel_utils, imported by the T5 wrapper).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pytorch-lightning>=2.2.1", "transformers==4.39.3", "mirdata", "mido",
                "deprecated", "smart-open", "einops", "wandb",
                "git+https://github.com/craffel/mir_eval.git",
                "git+https://github.com/katsura-jp/pytorch-cosine-annealing-with-warmup.git"], check=True)
print("installed YourMT3 training deps")

# --- config.py: data_home -> our Drive datasets (default ../../data); num_workers=1
#     (Google Drive FUSE throws Errno 5 under multiple parallel dataloader workers).
cfgpy = AMT_SRC / "config" / "config.py"
_cfg = cfgpy.read_text()
_cfg = re.sub(r'"data_home":\s*"[^"]*"', f'"data_home": "{DATA_HOME}"', _cfg, count=1)
_cfg = re.sub(r'"num_workers":\s*\d+', '"num_workers": 1', _cfg, count=1)
cfgpy.write_text(_cfg)
print("config.py: data_home ->", DATA_HOME, "| num_workers -> 1")

# --- YourMT3 source compat patches (idempotent; marker-guarded) --------------
def _patch(relpath, marker, old, new, note):
    p = AMT_SRC / relpath
    t = p.read_text()
    if marker in t:
        print(f"  {note}: already applied"); return
    assert old in t, f"anchor not found for '{note}' in {relpath} (YourMT3 source changed?)"
    p.write_text(t.replace(old, new, 1)); print(f"  {note}: patched")

# (1) attach a CSVLogger (Trainer ships with none) -> loss curves in §8
_patch("model/init_train.py", "restrudel CSVLogger",
       "    trainer = pl.trainer.trainer.Trainer(**train_params)",
       '    from pytorch_lightning.loggers import CSVLogger as _RS_CSV  # restrudel CSVLogger\n'
       '    train_params["logger"] = _RS_CSV(save_dir=lightning_dir, name="metrics")\n'
       "    trainer = pl.trainer.trainer.Trainer(**train_params)",
       "CSVLogger")

# (1b) log EVERY step + validate mid-run — otherwise short (smoke) runs produce an
#      empty loss curve: PL's default flushes train_loss every 50 steps and runs
#      validation only at epoch end, which a 10-step run never reaches.
_patch("model/init_train.py", "restrudel log-cadence",
       '    train_params["logger"] = _RS_CSV(save_dir=lightning_dir, name="metrics")',
       '    train_params["logger"] = _RS_CSV(save_dir=lightning_dir, name="metrics")\n'
       '    train_params["log_every_n_steps"] = 1  # restrudel log-cadence\n'
       '    if int(getattr(args, "max_steps", -1) or -1) > 0:\n'
       '        train_params["val_check_interval"] = max(1, int(args.max_steps) // 2)',
       "log cadence")

# (2) guard wandb_logger (None when wandb disabled)
_wp = AMT_SRC / "train.py"
_wt = _wp.read_text()
if "restrudel-wandb-guard" not in _wt:
    _wt = _wt.replace(
        "    if trainer.global_rank == 0:\n        wandb_logger.experiment.config.update",
        "    if trainer.global_rank == 0 and wandb_logger is not None:  # restrudel-wandb-guard\n        wandb_logger.experiment.config.update", 1)
    _wt = _wt.replace(
        "    wandb_logger.watch(model, log='gradients', log_freq=5000)",
        "    if wandb_logger is not None:\n        wandb_logger.watch(model, log='gradients', log_freq=5000)", 1)
    _wp.write_text(_wt); print("  wandb guard: patched")
else:
    print("  wandb guard: already applied")

# (3) torch>=2.6 defaults torch.load(weights_only=True); the official ckpt is trusted
for _f in ("train.py", "test.py"):
    _p = AMT_SRC / _f
    _t = _p.read_text()
    if 'torch.load(dir_info["last_ckpt_path"])' in _t:
        _p.write_text(_t.replace('torch.load(dir_info["last_ckpt_path"])',
                                 'torch.load(dir_info["last_ckpt_path"], weights_only=False)'))
        print(f"  torch.load weights_only: patched {_f}")

# (4) disable torch.compile in the Perceiver (inductor is memory-hungry -> OOM)
_patch("model/perceiver_mod.py", "restrudel-no-compile",
       "    def forward(self, **kwargs):\n        if self.training is True:\n            return self._forward_compile(**kwargs)\n        else:\n            return self._forward_no_compile(**kwargs)",
       "    def forward(self, **kwargs):  # restrudel-no-compile\n        return self._forward_no_compile(**kwargs)",
       "torch.compile off")

# (5) gradient checkpointing on the T5 stacks (default off) -> fits decoder in 40GB
_patch("model/t5mod.py", "restrudel-grad-ckpt",
       "        self.gradient_checkpointing = False",
       "        self.gradient_checkpointing = True  # restrudel-grad-ckpt",
       "gradient checkpointing")

# (6) with grad-ckpt use_cache=False, the decoder returns a 1-tuple -> take [0]
_patch("model/ymt3.py", "restrudel-dec-unpack",
       "        dec_hs, _ = self.decoder(inputs_embeds=dec_inputs_embeds, encoder_hidden_states=enc_hs, return_dict=False)",
       "        dec_hs = self.decoder(inputs_embeds=dec_inputs_embeds, encoder_hidden_states=enc_hs, return_dict=False)[0]  # restrudel-dec-unpack",
       "decoder unpack")
# (7) explicitly save final weights after fit: Lightning's ModelCheckpoint monitors
#     validation/macro_onset_f (never logged by our runs) so it silently never fires —
#     without this, last.ckpt stays the SEED and the "fine-tuned" model equals the base.
_patch("train.py", "restrudel-save-final",
       '        trainer.fit(model, ckpt_path=dir_info["last_ckpt_path"], datamodule=dm)',
       '        trainer.fit(model, ckpt_path=dir_info["last_ckpt_path"], datamodule=dm)\n\n'
       '    trainer.save_checkpoint(dir_info["lightning_dir"] + "/checkpoints/last.ckpt")  # restrudel-save-final',
       "final save")
# (8) retry transient Google Drive FUSE I/O errors: under sustained random reads the
#     Drive mount sporadically throws OSError(Errno 5) on a perfectly fine wav —
#     without a retry, ONE hiccup kills the whole multi-hour training run.
_ap = AMT_SRC / "utils" / "audio.py"
_at = _ap.read_text()
if "restrudel-io-retry" not in _at:
    _at += '''

# restrudel-io-retry: Google Drive FUSE sporadically raises OSError(Errno 5) /
# EOFError on valid files under load; retry with backoff (~1+2+4+8+16s) before
# giving up, instead of letting one hiccup abort a multi-hour training run.
import time as _rs_time


def _rs_io_retry(fn, _tries=6):
    def wrapped(*a, **k):
        for i in range(_tries):
            try:
                return fn(*a, **k)
            except (OSError, EOFError):
                if i == _tries - 1:
                    raise
                _rs_time.sleep(2 ** i)
    wrapped.__name__ = fn.__name__
    return wrapped


load_audio_file = _rs_io_retry(load_audio_file)
get_audio_file_info = _rs_io_retry(get_audio_file_info)
'''
    _ap.write_text(_at); print("  io retry: patched")
else:
    print("  io retry: already applied")
print("compat patches applied.")



## 4. Bench lists & presets

(a) Rebase strudel index paths onto this runtime's `DATA_HOME`. (b) Build one capped
test list per category — corpus holdout is the **true test**; synthetic batches use
their **validation** files (val-diag); references capped at `REF_CAP`. (c) Register the
`bench_<cat>` presets in YourMT3's `data_presets.py` (same marker-guarded block as
notebook 05, so running both in one runtime stays idempotent).


In [ ]:
# (a) rebase strudel index paths onto this DATA_HOME
SONG_ROOT = DATA_HOME / "strudel_yourmt3_16k"
for split in ("train", "validation", "test"):
    p = INDEX_DIR / f"strudel_{split}_file_list.json"
    fl = json.load(open(p)); n = 0
    for e in fl.values():
        sid = e.get("strudel_id")
        if not sid:
            continue
        d = SONG_ROOT / sid
        for key, fname in (("mix_audio_file", "mix.wav"), ("note_events_file", f"{sid}_note_events.npy"),
                           ("notes_file", f"{sid}_notes.npy"), ("midi_file", f"{sid}.mid")):
            if key in e and e[key] != str(d / fname):
                e[key] = str(d / fname); n += 1
    json.dump(fl, open(p, "w"), indent=4)
    print(f"rebased strudel {split}: {n} paths")

# (b) per-category benchmark test lists (strudel capped at EVAL_CAP, refs at REF_CAP)
def load_fl(path):
    return json.load(open(path)) if path.exists() else {}

bench_entries = defaultdict(list)
for e in load_fl(INDEX_DIR / "strudel_test_file_list.json").values():
    bench_entries[strudel_subcat(e["strudel_id"])].append(e)   # corpus holdout = TRUE test
# synthetic batches: no true test split BY DESIGN -> bench on their VALIDATION files
for e in load_fl(INDEX_DIR / "strudel_validation_file_list.json").values():
    _cat = strudel_subcat(e["strudel_id"])
    if _cat != "strudel_corpus":
        bench_entries[_cat].append(e)
for cat in ("maestro", "slakh", "egmd"):
    for e in load_fl(INDEX_DIR / f"{cat}_test_file_list.json").values():
        bench_entries[cat].append(e)

BENCH_N = {}
for cat in CATEGORIES:
    cap = (EVAL_CAP if cat.startswith("strudel") else REF_CAP)
    entries = bench_entries.get(cat, [])
    use = entries if FULL_EVAL else entries[:cap]
    json.dump({str(i): e for i, e in enumerate(use)},
              open(INDEX_DIR / f"bench_{cat}_test_file_list.json", "w"), indent=4)
    BENCH_N[cat] = len(use)
print("bench lists (files/category):", dict(BENCH_N))

# (c) register presets (identical marker-guarded block as notebook 05 -> idempotent)
DP = AMT_SRC / "config" / "data_presets.py"
VOCAB = {"maestro": ("[PIANO_SOLO_CLASS]", None, False),
         "slakh":   ("[GM_INSTR_CLASS]", 'drum_vocab_presets["gm"]', True),
         "egmd":    ("[None]", 'drum_vocab_presets["ksh"]', False)}
def strudel_vocab():
    return ("[GM_INSTR_CLASS]", 'drum_vocab_presets["gm"]', False)

def single_preset(name, dataset_name, ev, edv, has_stem, splits=("train", "validation", "test")):
    L = [f'    "{name}": {{', f'        "eval_vocab": {ev},']
    if edv:
        L.append(f'        "eval_drum_vocab": {edv},')
    L += [f'        "dataset_name": "{dataset_name}",',
          f'        "train_split": "{splits[0]}",',
          f'        "validation_split": "{splits[1]}",',
          f'        "test_split": "{splits[2]}",',
          f'        "has_stem": {has_stem},', '    },']
    return "\n".join(L)

src = DP.read_text()
if "restrudel presets v2" not in src:
    singles = ["    # >>> restrudel presets v2 >>>",
               single_preset("strudel", "strudel", "[GM_INSTR_CLASS]", 'drum_vocab_presets["gm"]', False)]
    for cat in CATEGORIES:                              # bench_<cat>: all splits point at the test list
        ev, edv, hs = strudel_vocab() if cat.startswith("strudel") else VOCAB[cat]
        singles.append(single_preset(f"bench_{cat}", f"bench_{cat}", ev, edv, hs,
                                     splits=("test", "test", "test")))
    singles.append("    # <<< restrudel presets v2 <<<")
    single_block = "\n".join(singles) + "\n"
    multi_block = "\n".join([
        "    # >>> restrudel strudel_ft v2 >>>",
        '    "strudel_ft": {',
        '        "presets": ["strudel", "slakh", "maestro", "egmd"],',
        f'        "weights": {list(WEIGHTS.values())},',
        "        \"eval_vocab\": [GM_INSTR_CLASS, None, None, None],",
        '        "eval_drum_vocab": drum_vocab_presets["gm"],',
        '        "val_max_num_files": 20,',
        '        "test_max_num_files": None,',
        "    },",
        "    # <<< restrudel strudel_ft v2 <<<"]) + "\n"
    src = src.replace("data_preset_single_cfg = {\n", "data_preset_single_cfg = {\n" + single_block, 1)
    src = src.replace("data_preset_multi_cfg = {\n", "data_preset_multi_cfg = {\n" + multi_block, 1)
    DP.write_text(src)
    print("registered strudel + bench presets")
else:
    print("presets already registered")
for nm in ["strudel"] + [f"bench_{c}" for c in CATEGORIES]:
    assert f'"{nm}"' in DP.read_text(), f"preset {nm} missing"
print("presets ready")

## 5. Checkpoints — restore from Drive

Base model from the Drive cache (or HF, one-time); each fine-tuned variant's
`last.ckpt` from its `YourMT3+_fine_tuned_*` Drive folder into the `amt/logs`
layout `test.py` expects.


In [ ]:
RELEASED_CKPT = MODEL_ROOT / "amt" / "logs" / "2024" / RELEASED_EXP / "checkpoints" / "last.ckpt"

def ensure_checkpoint():
    if RELEASED_CKPT.exists():
        print("base checkpoint already local"); return
    cache_ck = DRIVE_MODEL_CACHE / "amt" / "logs" / "2024" / RELEASED_EXP / "checkpoints" / "last.ckpt"
    if cache_ck.exists():
        print("restoring base checkpoint from Drive cache…")
        RELEASED_CKPT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cache_ck, RELEASED_CKPT)
    else:
        print("fetching base checkpoint from HF model repo (~1GB, one-time)…")
        snapshot_download(repo_id="mimbres/YourMT3", repo_type="model",
                          allow_patterns=[f"logs/2024/{RELEASED_EXP}/**"], local_dir=str(MODEL_ROOT / "amt"))
        cache_ck.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(RELEASED_CKPT, cache_ck)
    assert RELEASED_CKPT.exists(), "base checkpoint missing after ensure_checkpoint()"

ensure_checkpoint()
for v, info in VARIANTS.items():
    ck = MODEL_ROOT / "amt" / "logs" / "ymt3" / info["exp"] / "checkpoints" / "last.ckpt"
    if ck.exists():
        print(f"{v}: checkpoint already local")
    else:
        ck.parent.mkdir(parents=True, exist_ok=True)
        print(f"{v}: copying {info['dir'].name}/last.ckpt from Drive…")
        shutil.copy2(info["dir"] / "last.ckpt", ck)
    assert ck.exists()
print("all checkpoints ready:", ["base", *VARIANTS])

## 6. Run the benchmark — background driver

Each model × category run is a full `test.py` transcription pass (autoregressive
decode), so the whole sweep takes **hours** — it runs as a detached background driver
that survives notebook/browser disconnects. Fast strudel categories run first (headline
result early); metrics stream to `benchmark_metrics_raw.json` on Drive after every
model × category. Poll with the next cell.


In [ ]:
# 6a. launch the bench driver (detached; safe to disconnect afterwards)
assert torch and torch.cuda.is_available(), "no GPU — set Runtime > Change runtime type > GPU"
BENCH_ORDER = ["strudel_corpus", "strudel_synthetic_batch_1", "strudel_synthetic_batch_2",
               "strudel_synthetic_batch_3", "strudel_synthetic_batch_4",
               "egmd", "slakh", "maestro"]          # fast headline categories first
BENCH_LOGDIR = COMPARE_DIR / "bench_logs"; BENCH_LOGDIR.mkdir(parents=True, exist_ok=True)

BENCH_CFG = COMPARE_DIR / "bench_driver_config.json"
BENCH_CFG.write_text(json.dumps({
    "model_root": str(MODEL_ROOT), "arch": ARCH, "precision": PRECISION,
    "status": str(COMPARE_DIR / "bench_status.json"),
    "logdir": str(BENCH_LOGDIR), "raw_out": str(COMPARE_DIR / "benchmark_metrics_raw.json"),
    "order": BENCH_ORDER,
    "models": {m: list(t) for m, t in MODELS.items()}}, indent=2))

BENCH_DRIVER = (Path("/content") if IN_COLAB else REPO) / f"bench_driver_{TIMESTAMP}.py"
BENCH_DRIVER.write_text(r'''
import json, os, re, shutil, subprocess, sys, time, datetime
from pathlib import Path
cfg = json.loads(Path(sys.argv[1]).read_text())
MODEL_ROOT = Path(cfg["model_root"]); STATUS = Path(cfg["status"]); LOGDIR = Path(cfg["logdir"])
RAW = Path(cfg["raw_out"])
env = dict(os.environ); env.update({"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "TORCHDYNAMO_DISABLE": "1"})
state = {"phase": "benchmarking", "started": datetime.datetime.now().isoformat(timespec="seconds"),
         "models": list(cfg["models"]), "runs": {}}
def save(): STATUS.write_text(json.dumps(state, indent=2))
save()
all_m = json.loads(RAW.read_text()) if RAW.exists() else {}     # resume partial sweeps
for cat in cfg["order"]:
    all_m.setdefault(cat, {})
    for mname, (exp_id, project) in cfg["models"].items():
        key = "%s/%s" % (mname, cat)
        if all_m[cat].get(mname):
            state["runs"][key] = "cached (already done)"; save(); continue
        t0 = time.time()
        state["runs"][key] = "running"; save()
        exp_dir = MODEL_ROOT / "amt" / "logs" / project / exp_id
        cmd = [sys.executable, "amt/src/test.py", exp_id, "-p", project, "-d", "bench_" + cat,
               *cfg["arch"], "-pr", cfg["precision"], "-g", "auto"]
        log = exp_dir / ("bench_%s_stdout.log" % cat)
        with open(log, "w") as f:
            r = subprocess.run(cmd, cwd=MODEL_ROOT, stdout=f, stderr=subprocess.STDOUT, env=env)
        shutil.copy2(log, LOGDIR / ("%s_%s.log" % (mname, cat)))
        m = {}
        for rf in sorted(exp_dir.glob("result_*_bench_%s.json" % cat)):
            for name, val in re.findall(r"\'test/\([^)]*\)([^\']+)\':\s*(?:tensor\()?([0-9.]+)", rf.read_text()):
                m[name] = float(val)
            shutil.copy2(rf, LOGDIR / ("%s_%s_result.json" % (mname, cat)))
        if not m:
            for name, val in re.findall(r"test/\([^)]*\)(\w+)\s*│\s*([\d.]+)", log.read_text()):
                m[name] = float(val)
        all_m[cat][mname] = m
        RAW.write_text(json.dumps(all_m, indent=2))             # progressive save to Drive
        state["runs"][key] = "%s: %d metrics in %ds" % (
            "done" if r.returncode == 0 else "EXIT %d" % r.returncode, len(m), round(time.time() - t0))
        save()
state["phase"] = "finished"; state["finished"] = datetime.datetime.now().isoformat(timespec="seconds"); save()
''')
_proc = subprocess.Popen([sys.executable, str(BENCH_DRIVER), str(BENCH_CFG)],
                         stdout=open(COMPARE_DIR / "bench_driver_stdout.log", "w"), stderr=subprocess.STDOUT)
print(f"bench driver started (pid {_proc.pid}) — models={list(MODELS)}")
print("status: ", COMPARE_DIR / "bench_status.json")
print("metrics:", COMPARE_DIR / "benchmark_metrics_raw.json  (updates after every model x category)")

In [ ]:
# 6b. poll benchmark progress (re-run anytime)
_st = COMPARE_DIR / "bench_status.json"
if not _st.exists():
    print("no bench status yet — did 6a run?")
else:
    _s = json.loads(_st.read_text())
    print(f"phase: {_s.get('phase')}   started: {_s.get('started')}   finished: {_s.get('finished', '—')}")
    for k, v in _s.get("runs", {}).items():
        print(f"  {k:<40} {v}")
    if _s.get("phase") == "finished":
        print("\nBENCHMARK FINISHED — continue with §7 (charts).")

## 7. Results — comparison charts + JSON

All numbers are **note-level F1** (`onset_f` preferred, then `multi_f`). Synthetic
categories are marked **val-diag**. Charts + `comparison_results.json` go to the run's
Drive folder for later custom visualization.


In [ ]:
# load the raw metrics written by the driver (works in a fresh session)
RAW = COMPARE_DIR / "benchmark_metrics_raw.json"
assert RAW.exists(), "no benchmark_metrics_raw.json yet — run §6 first"
BENCH = {c: m for c, m in json.load(open(RAW)).items() if m}
print("categories with results:", {c: list(m) for c, m in BENCH.items()})

MODEL_COLORS = {"base": "#8d99ae", "strudel50": "#d1495b", "egmd50": "#2e4057"}
_extra = [c for c in ("#66a182", "#e36414", "#7768ae") ]
for _m in MODELS:
    if _m not in MODEL_COLORS:
        MODEL_COLORS[_m] = _extra.pop(0) if _extra else None

def primary(metrics):
    for pref in ("onset_f", "multi_f", "frame_f", "onset_f_drum"):  # drum-only cats (egmd) have no pitched onset_f
        if pref in metrics:
            return metrics[pref]
    return next(iter(metrics.values()), float("nan"))

# --- (1) per dataset category -------------------------------------------------
cats = [c for c in BENCH]
print(f"{'category':<28}" + "".join(f"{m:>12}" for m in MODELS))
for c in cats:
    print(f"{c:<28}" + "".join(f"{primary(BENCH[c].get(m, {})):>12.3f}" for m in MODELS))

x = np.arange(len(cats)); w = 0.8 / len(MODELS)
fig, ax = plt.subplots(figsize=(max(9, 2 * len(cats)), 5))
for j, m in enumerate(MODELS):
    ax.bar(x + (j - (len(MODELS) - 1) / 2) * w, [primary(BENCH[c].get(m, {})) for c in cats], w,
           label=m, color=MODEL_COLORS.get(m))
ax.set_xticks(x)
ax.set_xticklabels([c + (" · val-diag" if "synthetic" in c else "") for c in cats],
                   rotation=20, ha="right", fontsize=8)
ax.set_ylabel("note F1 (primary)"); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("All models × dataset categories")
plt.tight_layout(); fig.savefig(COMPARE_DIR / "comparison_categories.png", dpi=120); plt.show()

# --- (2) per instrument group on the TARGET (strudel_corpus) -------------------
# groups matched by substring against per-class metric names (onset_f_<Class>);
# 'effects' has no note class (FX change timbre, not notes) — shown as n/a.
GROUPS = OrderedDict([
    ("drums",           ["drum"]),
    ("synth leads",     ["Synth Lead"]),
    ("bass",            ["Bass"]),
    ("chords (pads/keys)", ["Synth Pad", "Piano", "Organ"]),
    ("effects",         []),
])
def group_score(metrics, pats):
    vals = [v for k, v in metrics.items()
            if k.startswith("onset_f") and any(p.lower() in k.lower() for p in pats)]
    return float(np.mean(vals)) if vals else float("nan")

tgt = "strudel_corpus"
if tgt in BENCH:
    print(f"\nInstrument groups on {tgt} (mean onset-F1 over matching classes):")
    print(f"{'group':<20}" + "".join(f"{m:>12}" for m in MODELS))
    gvals = {m: [group_score(BENCH[tgt].get(m, {}), pats) for pats in GROUPS.values()] for m in MODELS}
    for i, g in enumerate(GROUPS):
        print(f"{g:<20}" + "".join(f"{gvals[m][i]:>12.3f}" for m in MODELS))
    x = np.arange(len(GROUPS))
    fig, ax = plt.subplots(figsize=(10, 5))
    for j, m in enumerate(MODELS):
        ax.bar(x + (j - (len(MODELS) - 1) / 2) * w, gvals[m], w, label=m, color=MODEL_COLORS.get(m))
    ax.set_xticks(x); ax.set_xticklabels(list(GROUPS), rotation=15, ha="right", fontsize=8)
    ax.set_ylabel("mean onset F1"); ax.legend(); ax.grid(axis="y", alpha=0.3)
    ax.set_title(f"Instrument groups on {tgt} — 'effects' has no note class (n/a)")
    plt.tight_layout(); fig.savefig(COMPARE_DIR / "comparison_instrument_groups.png", dpi=120); plt.show()

# --- persist EVERYTHING for later custom viz ----------------------------------
results = {
    "run": RUN_NAME, "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "models": {m: {"exp": MODELS[m][0], "project": MODELS[m][1],
                   "weights": VARIANTS[m]["weights"] if m in VARIANTS else None} for m in MODELS},
    "eval_cap": EVAL_CAP, "ref_cap": REF_CAP, "full_eval": FULL_EVAL,
    "n_test_files": {c: BENCH_N.get(c) for c in cats},
    "metrics": BENCH,
    "instrument_groups": {g: pats for g, pats in GROUPS.items()},
}
(COMPARE_DIR / "comparison_results.json").write_text(json.dumps(results, indent=2, default=str))
print("saved ->", COMPARE_DIR / "comparison_results.json")

print(f"\n=== all artifacts in Google Drive: {COMPARE_DIR} ===")
for f in sorted(COMPARE_DIR.rglob("*")):
    if f.is_file():
        print(f"  {str(f.relative_to(COMPARE_DIR)):<40} {f.stat().st_size/1e6:8.2f} MB")